# MOEA/D — full experiment and tuning alignment with NSGA-II (IDS 560)
Run in order. Exp 1–6 reuse verified results when present. Both scikit-learn (36 runs) and Optuna (24 trials, 9 reseeds, 3 baseline runs) are enabled by default. This may be computationally intensive. No tuning files are claimed complete until the final audits succeed.


In [ ]:
%pip -q install numpy pandas scikit-learn optuna matplotlib


## 1 — Upload and extract the project ZIP
The ZIP contains a top-level `MOEAD_Complete_Project` directory. This cell finds it and changes the working directory, so the next cells resolve Python scripts correctly.


In [ ]:
from google.colab import files
from pathlib import Path
import os, json, sys, subprocess, zipfile
archives=list(Path('/content').glob('*MOEAD*.zip'))
if not archives:
    print('Upload MOEAD_Reviewed_Aligned.zip or MOEAD_Complete_Project(1).zip')
    files.upload()
    archives=list(Path('/content').glob('*MOEAD*.zip'))
assert archives, 'No MOEA/D project zip found in /content'
archive=next((p for p in archives if p.name=='MOEAD_Full_Aligned_With_Results.zip'),archives[-1])
workspace=Path('/content/moead_reviewed')
workspace.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(workspace)
projects=[x.parent for x in workspace.rglob('moead_core.py')]
assert len(projects)==1, f'Expected one moead_core.py, found {len(projects)}'
project=projects[0]
assert (project/'verify_tuning_outputs.py').is_file(), 'Upload the latest MOEAD_Full_Aligned_With_Results.zip, not the earlier ZIP.'
os.chdir(project)
csv=Path('synthetic_fraud_500k.csv')
if not csv.is_file():
    print('CSV missing from ZIP; upload the exact shared 500K dataset')
    files.upload()
assert csv.is_file(), 'Missing data CSV'
print('Project:',Path.cwd(),'CSV:',csv,'size MB:',round(csv.stat().st_size/1e6,1))


## 2 — Verify preprocessing on the actual dataset
This checks that the **source definitions** of NSGA-II and MOEA/D produce the same ordered conditions. To verify the NSGA-II teammate actually used the *same data*, ask them for the source CSV SHA-256 and compare it to this report.


In [ ]:
subprocess.run([sys.executable,'compare_preprocessing.py','--csv',str(csv),'--out','data_parity_report.json'],check=True)
r=json.load(open('data_parity_report.json'))
print('CSV SHA-256:',r['csv_sha256'])
print('Conditions SHA-256:',r['condition_bank_sha256'])
print('Split sizes:',r['split_sizes'],'Frauds:',r['split_fraud_counts'])


## 3 — Run exact Exp 1–6 settings
Set `EXPERIMENTS='all'` to run all six using the configs in the team's NSGA-II `RUN_COMMANDS.txt`. Exp 4 is already in the ZIP. `SKIP_MATCHING_EXISTING=True` avoids rerunning complete settings but **does not independently certify** a teammate's dataset SHA.


In [ ]:
EXPERIMENTS='all'  # alternatives: '1', '2', '3', '4', '5', '6'
SKIP_MATCHING_EXISTING=True
cmd=[sys.executable,'run_experiments.py','--csv',str(csv),'--exp',EXPERIMENTS]
if SKIP_MATCHING_EXISTING:cmd.append('--skip-existing')
subprocess.run(cmd,check=True)


## 4 — Scikit-learn-style parameter search (12 configurations × 3 seeds)
This imports the teammate's **byte-identical** `param_search.py`; fixed population 200, cap 160, same discrete grid, random state 0, seeds 0/1/2, score = validation HV. Enable to generate actual outputs; otherwise it stays incomplete.


In [ ]:
RUN_PARAMETER_SEARCH=True  # Enabled: 36 real runs
if RUN_PARAMETER_SEARCH:
    subprocess.run([sys.executable,'moead_search.py','--csv',str(csv),
      '--n_iter','12','--seeds','3','--pop','200','--jobs','2',
      '--out','results/parameter_search/moead_search'],check=True)
else:print('Skipped parameter search — no results were generated by this cell.')


## 5 — Optuna TPE search (24 trials + top-3 reseeding + Exp-4 baseline)
Matches NSGA-II search space and trial count, TPE seed 0, startup trials 8, fixed pop 200. Best three trials are evaluated again with seeds 101, 102, 103. Enable to obtain the full results. Because TPE is adaptive, **MOEA/D and NSGA-II need not select the same continuous hyperparameter values**. Two parallel SQLite workers may not reproduce a fixed trial order across runs.


In [ ]:
RUN_OPTUNA=True  # Enabled: 24 trials, 9 reseeds, 3 baseline seeds
if RUN_OPTUNA:
    subprocess.run([sys.executable,'moead_optuna.py','--csv',str(csv),
       '--trials','24','--pop','200','--workers','2',
       '--out','results/optuna'],check=True)
    subprocess.run([sys.executable,'moead_optuna_baseline.py','--csv',str(csv),
       '--pop','200','--gens','250','--cx','0.9','--mut','0.8',
       '--out','results/optuna/exp4_setting_same_seeds.csv'],check=True)
else:print('Skipped Optuna — no Optuna results were generated by this cell.')


## Shared Random Forest benchmark (same NSGA-II code and 500K dataset)
The RF is a **shared model-independent benchmark** for NSGA-II, SPEA2 and MOEA/D, not a separate MOEA/D algorithm. This exact script produces `rf_benchmark_results.csv` and `rf_feature_importance.csv`. Existing results are reused.


In [ ]:
rf_dir=Path('results/rf_benchmark')
rf_dir.mkdir(parents=True,exist_ok=True)
if not (rf_dir/'rf_benchmark_results.csv').is_file() or not (rf_dir/'rf_feature_importance.csv').is_file():
    subprocess.run([sys.executable,str(Path.cwd()/'rf_benchmark.py'),str(Path.cwd()/'synthetic_fraud_500k.csv')],cwd=rf_dir,check=True)
else:
    print('RF benchmark already present — not retraining')


## 6 — Generate figures and verify completeness
Figures are made only for experiments that actually have results. The audit prints what's complete or missing for each experiment and tuning protocol.


In [ ]:
subprocess.run([sys.executable,'make_moead_plots.py'],check=True)
subprocess.run([sys.executable,'verify_completion.py','--csv',str(csv)],check=True)
subprocess.run([sys.executable,'plot_optuna_search.py'],check=True)
subprocess.run([sys.executable,'verify_tuning_outputs.py','results'],check=True)


## 7 — Download results ZIP
Optional: bundle results, reports and plots. The raw 500K CSV is excluded from the results ZIP to keep it small.


In [ ]:
from shutil import make_archive
from google.colab import files
from zipfile import ZipFile,ZIP_DEFLATED
# Before download, insist the searches have actually finished.
subprocess.run([sys.executable,'verify_tuning_outputs.py','results'],check=True)
result_zip=make_archive('/content/MOEAD_results_COMPLETE','zip',root_dir=Path.cwd(),base_dir='results')
print('Results ZIP:', result_zip)
files.download(result_zip)
# This second ZIP also includes source code and README for teammate review; raw CSV excluded.
full_zip=Path('/content/MOEAD_code_and_results_COMPLETE.zip')
with ZipFile(full_zip,'w',compression=ZIP_DEFLATED) as z:
    for p in Path.cwd().rglob('*'):
        if p.is_file() and p.name!='synthetic_fraud_500k.csv' and '__pycache__' not in p.parts and '.ipynb_checkpoints' not in p.parts:
            z.write(p, arcname=str(Path('MOEAD_Complete_Project')/p.relative_to(Path.cwd())))
print('Code + results ZIP:',full_zip)
files.download(str(full_zip))
